#### The Core idea is to wrap our training and evaluation logic in an objective funtion that optuna can call

In [4]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import optuna

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = 28
sequence_length = 28
n_classes = 10

# Use a subset of data to make trials faster
dataset_subset = 5000

In [6]:
transform = transforms.ToTensor()

train_set = datasets.MNIST(root = "./data3", train = True, transform = transform, download = True)
val_set = datasets.MNIST(root = "./data3", train = False, transform = transform, download = True)

In [9]:
# Create subsets
train_subset = Subset(dataset = train_set, indices = range(dataset_subset))
val_subset = Subset(dataset = val_set, indices = range(dataset_subset // 5))

In [10]:
class RecurrentNet(nn.Module):
    def __init__(self, hidden_size, n_layers):
        super(RecurrentNet, self).__init__()
        self.recurrent_layer = nn.LSTM(input_size = input_size, hidden_size = hidden_size, num_layers = n_layers, batch_first = True)
        self.fc = nn.Linear(in_features = hidden_size, out_features = n_classes)
        
    def forward(self, x):
        x = x.squeeze(1)
        out, _ = self.recurrent_layer(x)
        out = self.fc(out[:, -1, :])
        return out

In [14]:
# The Objective funtion for Optuna
# This function takes a "trial" object, defines the hyperparameters, trains a model, and returns the validation accuracy

def objective(trial):
    # Define Hyperparameters to search
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log = True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    
    # Setup Model, DataLoaders, Optimizer
    model = RecurrentNet(hidden_size = hidden_size, n_layers = num_layers).to(device)
    train_loader = DataLoader(train_subset, batch_size = batch_size, shuffle = True)
    val_loader = DataLoader(val_subset, batch_size = batch_size, shuffle = False)
    
    optimizer = getattr(torch.optim, optimizer_name)(params = model.parameters(), lr = lr)
    criterion = nn.CrossEntropyLoss()
    
    # Training and Validation Loop(for a fixed number of epochs)
    num_epochs = 2
    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
    # Evaluate on the validation set
    model.eval()
    n_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            n_correct += (predicted == labels).sum().item()
            
    accuracy = n_correct / len(val_loader.dataset)
    
    # Optuna will try to maximize this value
    return accuracy

In [17]:
# Run the Hyperparameter search
# A study is a collection of trials for an optimization task.
# We specify "direction = "maximize" " because we want the highest accuracy
study = optuna.create_study(direction = "maximize")

#  Start the optimization. Optuna will call the 'objective' function 30 times
study.optimize(func = objective, n_trials = 30)

[I 2025-08-07 18:20:04,944] A new study created in memory with name: no-name-e822ab45-d3a0-4bbc-a752-ea1fa5b8aa25
[I 2025-08-07 18:20:06,040] Trial 0 finished with value: 0.18 and parameters: {'hidden_size': 64, 'num_layers': 2, 'lr': 0.00011729010915870395, 'batch_size': 128, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.18.
[I 2025-08-07 18:20:07,411] Trial 1 finished with value: 0.126 and parameters: {'hidden_size': 128, 'num_layers': 3, 'lr': 0.001969983357721206, 'batch_size': 64, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.18.
[I 2025-08-07 18:20:08,298] Trial 2 finished with value: 0.087 and parameters: {'hidden_size': 64, 'num_layers': 1, 'lr': 0.004042455835965824, 'batch_size': 128, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.18.
[I 2025-08-07 18:20:09,214] Trial 3 finished with value: 0.169 and parameters: {'hidden_size': 64, 'num_layers': 1, 'lr': 0.0004147352997853054, 'batch_size': 128, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.18.
[I 

In [18]:
print("\n--- Hyperparameter Search Finished ---")
print(f"Number of finished trials: {len(study.trials)}")

print("\nBest trial:")
trial = study.best_trial
print(f"  Value (Validation Accuracy): {trial.value:.4f}")

print("\n  Best Parameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


--- Hyperparameter Search Finished ---
Number of finished trials: 30

Best trial:
  Value (Validation Accuracy): 0.8800

  Best Parameters:
    hidden_size: 128
    num_layers: 2
    lr: 0.0020039198086502924
    batch_size: 32
    optimizer: Adam
